# CareLoop Clinics – No-Show Guard
## Part 4 — GenAI: turn call-centre notes into structured reasons, then recommend

**Inputs:** `data/followup_notes.csv`, `appointments_clean.csv` · **Output:** `notes_analysed.csv`

The API key is typed at run time with `getpass` (hidden, never printed, never saved in this notebook).
Free tier is small, so this notebook is designed to use about 12 API calls in total.

In [ ]:
# --- Setup: works in Google Colab (uses Google Drive) and locally ---
import os

try:
    from google.colab import drive          # only exists inside Colab
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/CareLoop-NoShow-Guard"
    os.makedirs(os.path.join(PROJECT_DIR, "data"), exist_ok=True)
    os.chdir(PROJECT_DIR)                    # all relative paths start here
except ImportError:
    pass                                     # running locally: use the current folder

print("Working directory:", os.getcwd())

In [ ]:
!pip install -q google-genai pydantic

In [ ]:
import json
import time
import pandas as pd
from getpass import getpass
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

# Use the Flash model from your Session 22.1 notebook. If it has been retired,
# pick the current Flash model listed in Google AI Studio.
MODEL_NAME = "gemini-3.5-flash"
USE_THINKING_CONFIG = True      # set to False only if the call fails with an error about thinking_config

API_KEY = getpass("Paste your Gemini API key (hidden): ")
client = genai.Client(api_key=API_KEY)
del API_KEY                      # the key is no longer kept in a variable

API_CALLS = 0                    # counts every request actually sent to Gemini
ALLOWED_ROOT_CAUSES = ["forgot_or_no_reminder", "distance_or_transport", "cost_or_fee",
                       "illness_or_emergency", "work_or_schedule", "other"]

## Task 4.1 — One validated call

The prompt follows **CRAFT**: **C**ontext (who CareLoop is, what the note is), **R**ole (patient-experience analyst),
**A**ction (pick exactly one allowed root cause, summarise, recommend), **T**one (neutral, factual), **F**ormat (JSON with four fields).
`root_cause` is a plain string in the schema; the allowed values are in the prompt and enforced afterwards in Task 4.2.

In [ ]:
SCHEMA = {
    "type": "object",
    "properties": {"root_cause": {"type": "string"},
                   "confidence": {"type": "number"},
                   "summary": {"type": "string"},
                   "recommended_action": {"type": "string"}},
    "required": ["root_cause", "confidence", "summary", "recommended_action"],
}

SYSTEM_TEXT = (
    "You are a careful patient-experience analyst for an outpatient clinic chain in India. "
    "You read call-centre notes and return only the requested JSON. "
    "Never invent facts that are not in the note."
)

def build_craft_prompt(note_text):
    """V2 prompt: Context, Role, Action, Tone, Format."""
    return f"""CONTEXT: CareLoop Clinics runs eight outpatient clinics in India. After a patient misses an appointment,
the call centre phones them and writes a short note. Notes may be in English, Hindi or Hinglish.

ROLE: You are a patient-experience analyst classifying why the appointment was missed.

ACTION:
1. Choose exactly ONE root_cause from this list, written exactly as shown:
   forgot_or_no_reminder, distance_or_transport, cost_or_fee, illness_or_emergency, work_or_schedule, other
2. If the note does not state why the patient missed the visit (for example the patient could not be reached),
   answer "other". Do not guess.
3. Give confidence as a number between 0 and 1.
4. Write a summary in English of at most 25 words.
5. Suggest one concrete recommended_action for the clinic.

TONE: Neutral, factual and brief.

FORMAT: JSON with exactly these fields: root_cause, confidence, summary, recommended_action.

NOTE:
{note_text}"""

def make_config(system_text):
    settings = dict(temperature=0, max_output_tokens=1000,
                    response_mime_type="application/json", response_schema=SCHEMA)
    if system_text:
        settings["system_instruction"] = system_text
    if USE_THINKING_CONFIG:
        settings["thinking_config"] = types.ThinkingConfig(thinking_level="minimal")
    return types.GenerateContentConfig(**settings)

LAST_RAW_RESPONSE = ""          # the exact text of the most recent reply (kept for raw_response)

def ask_gemini(prompt, system_text):
    """Send one request and return the reply text exactly as received."""
    global API_CALLS, LAST_RAW_RESPONSE
    LAST_RAW_RESPONSE = ""
    API_CALLS += 1
    response = client.models.generate_content(model=MODEL_NAME, contents=prompt,
                                              config=make_config(system_text))
    if response.text is None:
        raise ValueError("empty reply from the model")
    LAST_RAW_RESPONSE = response.text
    return response.text

def analyse_note(note_text):
    """Classify one note with the CRAFT prompt and return the parsed dictionary."""
    raw = ask_gemini(build_craft_prompt(note_text), SYSTEM_TEXT)
    return json.loads(raw)

In [ ]:
notes = pd.read_csv("data/followup_notes.csv")
n001_text = notes.loc[notes["note_id"] == "N001", "note_text"].iloc[0]
print("N001:", n001_text)

n001_result = analyse_note(n001_text)
print("\nParsed dictionary:")
print(n001_result)
print("Fields:", sorted(n001_result.keys()))
print("API calls so far:", API_CALLS)

## Task 4.2 — Validate and process the first 10 notes

* Blank notes are detected **before** any API call (`pd.isna` first, because a blank note is read as NaN, a float with no `.strip()`).
* Every reply is kept exactly as returned in `raw_response`; invalid replies are recorded, never repaired.
* `try/except` makes sure one bad reply cannot stop the loop; `time.sleep(13)` keeps under the free-tier rate limit.

In [ ]:
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

def root_cause_allowed(value):
    return value in ALLOWED_ROOT_CAUSES

FIELDS = ["root_cause", "confidence", "summary", "recommended_action"]
records = []

for i, note in notes.head(10).iterrows():
    note_text = note["note_text"]
    record = {"note_id": note["note_id"], "appointment_id": note["appointment_id"]}

    # Input validation: missing, empty or only spaces -> no API call
    if pd.isna(note_text) or str(note_text).strip() == "":
        record.update(root_cause="other", confidence=0.0, summary="No note recorded",
                      recommended_action="Re-call patient", validation_status="skipped_blank", raw_response="")
        print(f"{note['note_id']}: blank note -> skipped_blank (no API call)")
        records.append(record)
        continue

    time.sleep(13)                           # respect the free-tier rate limit between calls

    data = None
    try:
        data = analyse_note(note_text)
        FollowupInsight(**data)              # Pydantic validation (raises ValidationError)
        if not root_cause_allowed(data["root_cause"]):
            raise ValueError(f"root_cause {data['root_cause']!r} is not in the allowed list")
        status = "valid"
    except Exception as error:               # API error, bad JSON, ValidationError or not-allowed value
        status = "invalid"
        print(f"{note['note_id']}: INVALID -> {type(error).__name__}: {error}")

    # Copy whatever fields the reply contained - never repair or fill in
    for field in FIELDS:
        record[field] = data.get(field) if isinstance(data, dict) else None
    record["validation_status"] = status
    record["raw_response"] = LAST_RAW_RESPONSE
    records.append(record)
    print(f"{note['note_id']}: {status} -> {record['root_cause']} (confidence {record['confidence']})")

results = pd.DataFrame(records, columns=["note_id", "appointment_id", "root_cause", "confidence", "summary",
                                         "recommended_action", "validation_status", "raw_response"])
results.to_csv("notes_analysed.csv", index=False)
print("\nSaved notes_analysed.csv:", results.shape, "| API calls so far:", API_CALLS)
results[["note_id", "root_cause", "confidence", "validation_status"]]

In [ ]:
# Agreement with the expected labels from the brief (for checking only - nothing is changed)
expected = {"N001": "forgot_or_no_reminder", "N002": "distance_or_transport", "N003": "cost_or_fee",
            "N004": "other", "N005": "illness_or_emergency", "N006": "work_or_schedule",
            "N007": "forgot_or_no_reminder", "N008": "distance_or_transport", "N009": "other",
            "N010": "work_or_schedule"}
check = results[results["validation_status"] != "skipped_blank"].copy()
check["expected"] = check["note_id"].map(expected)
check["agrees"] = check["root_cause"] == check["expected"]
print(f"Agreement on non-blank notes: {check['agrees'].sum()} of {len(check)}")
check[["note_id", "root_cause", "expected", "agrees"]]

## Task 4.3 — Break it on purpose
### (a) Two deliberately bad replies

In [ ]:
bad_replies = {
    "confidence = 1.7": {"root_cause": "cost_or_fee", "confidence": 1.7,
                         "summary": "Patient found the fee too high.", "recommended_action": "Offer a follow-up discount."},
    "missing recommended_action": {"root_cause": "cost_or_fee", "confidence": 0.8,
                                   "summary": "Patient found the fee too high."},
}
for label, reply in bad_replies.items():
    try:
        FollowupInsight(**reply)
        print(f"{label}: passed (unexpected)")
    except ValidationError as error:
        print(f"--- {label}: caught ValidationError ---")
        print(error, "\n")

### (b) Prompt comparison on N007 (Hinglish) and N009 (ambiguous)
Same model, schema and settings (JSON mode, temperature 0); **only the prompt differs**.
V1 = just "Classify this note". V2 = the CRAFT prompt — its replies for N007 and N009 are reused from Task 4.2 (identical request at temperature 0), which saves 2 calls.

In [ ]:
comparison_rows = []
for note_id in ["N007", "N009"]:
    note_text = notes.loc[notes["note_id"] == note_id, "note_text"].iloc[0]

    # V1: minimal prompt, no system instruction, same schema/settings
    time.sleep(13)
    try:
        v1_raw = ask_gemini(f"Classify this note: {note_text}", None)
        v1_cause = json.loads(v1_raw).get("root_cause")
    except Exception as error:
        v1_raw, v1_cause = LAST_RAW_RESPONSE, f"ERROR: {error}"
    comparison_rows.append({"note_id": note_id, "prompt": "V1 'Classify this note'",
                            "root_cause": v1_cause, "allowed": root_cause_allowed(v1_cause)})

    # V2: CRAFT prompt (reply from Task 4.2)
    v2_cause = results.loc[results["note_id"] == note_id, "root_cause"].iloc[0]
    comparison_rows.append({"note_id": note_id, "prompt": "V2 CRAFT",
                            "root_cause": v2_cause, "allowed": root_cause_allowed(v2_cause)})

prompt_table = pd.DataFrame(comparison_rows)
prompt_table

**Which prompt to deploy:** V2 (CRAFT). *(Write one or two sentences here after looking at the table above.)*
Points to check: did V1 return labels from the allowed list, or its own wording (which cannot be counted or merged)?
Did V2 return an allowed label for both notes, with `other` for N009 where the note gives no reason?

### (c) API call count

In [ ]:
print("Total Gemini API calls in Tasks 4.1-4.3:", API_CALLS)
print("  4.1: 1 (N001) | 4.2: 9 (10 notes minus blank N004) | 4.3: 2 (V1 for N007, N009; V2 reused)")
assert API_CALLS < 25

## Task 4.4 — Combine with the appointment data

In [ ]:
appointments = pd.read_csv("appointments_clean.csv")
merged = pd.read_csv("notes_analysed.csv").merge(appointments, on="appointment_id", how="left")
print("Merged rows:", len(merged))

print("\nroot_cause x reminder_sent:")
ct = pd.crosstab(merged["root_cause"], merged["reminder_sent"], margins=True)
print(ct)

print("\nNotes whose appointment had reminder_sent = No:", (merged["reminder_sent"] == "No").sum(), "of", len(merged))
forgot = merged[merged["root_cause"] == "forgot_or_no_reminder"]
print("forgot_or_no_reminder notes:", forgot["note_id"].tolist(),
      "| with reminder_sent = No:", (forgot["reminder_sent"] == "No").sum(), "of", len(forgot))
print("\nRoot-cause counts:")
print(merged["root_cause"].value_counts())

**Before downloading this notebook:** use *Edit → Find* to search the whole notebook for the first characters of your API key. It should appear nowhere (the key was typed into a hidden `getpass` box and never printed).